# Supplementary Results 11.4-11.5 — External check and the PAV × pleiotropy interaction

The criterion, the stratified contrasts and the non-linearity, repeated in the Pharmaprojects
target-indication data as processed for Minikel et al., and the interaction between PAV support and
pleiotropy fitted explicitly in both resources.

Numbers are written to `results/sr11_external.json`. Supplementary Results 11.1-11.3 are in
`11_criterion_validation.ipynb`.

**Provenance.** `chapters/_legacy/06-review-r1/or10-optimism-validation/`, notebooks 04, 05 and 06.
Pharmaprojects is a licensed resource: `ti_pairs_pharmaprojects` is built from files committed under
`chapters/_legacy/05-other-drug-indication-data/` that must not be redistributed (GAPS.md §1).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

from manuscript_methods import enrichment, paper

numbers = {}
PERMUTATIONS = 10000
SEED = 20260811

chembl = pd.read_parquet(paper.derived("ti_pairs_chembl"))
pharmaprojects = pd.read_parquet(paper.derived("ti_pairs_pharmaprojects"))
for frame in (chembl, pharmaprojects):
    frame["ta"] = frame["uniqueTherapeuticAreas"].fillna(0).astype(float)
    frame["gps"] = frame["uniqueDiseases"].fillna(0).astype(float)
    frame["support_all"] = enrichment.support_mask(frame).astype(int)
    frame["support_pav"] = enrichment.support_mask(frame, pav=True).astype(int)

DATASETS = [("ChEMBL", chembl), ("Pharmaprojects", pharmaprojects)]
print(f"ChEMBL pairs {len(chembl):,} | Pharmaprojects pairs {len(pharmaprojects):,}")

ChEMBL pairs 37,377 | Pharmaprojects pairs 7,390


## What the Pharmaprojects table holds

In [2]:
phases = pharmaprojects["maxClinicalPhase"].value_counts().sort_index()
print(phases.to_string())

numbers["S11.55"] = len(pharmaprojects)
numbers["S11.56"] = int((pharmaprojects["maxClinicalPhase"] == 1).sum())
numbers["S11.57"] = int((pharmaprojects["maxClinicalPhase"] == 2).sum())
numbers["S11.58"] = int((pharmaprojects["maxClinicalPhase"] == 3).sum())
numbers["S11.59"] = int(pharmaprojects["approved"].sum())

launched = set(map(tuple, pharmaprojects.loc[pharmaprojects["approved"] == 1, ["targetId", "diseaseId"]].values))
phase_four = set(map(tuple, chembl.loc[chembl["approved"] == 1, ["targetId", "diseaseId"]].values))
numbers["S11.60"] = len(launched & phase_four)
numbers["S11.61"] = round(100 * len(launched & phase_four) / len(launched), 0)
print(f"launched pairs also Phase IV in ChEMBL: {numbers['S11.60']} of {len(launched)} ({numbers['S11.61']}%)")

maxClinicalPhase
1    2274
2    3303
3     900
4     913
launched pairs also Phase IV in ChEMBL: 447 of 911 (49.0%)


## Our genetic support, and the resource's own annotation

In [3]:
ours = enrichment.or_rs(enrichment.support_mask(pharmaprojects), pharmaprojects["approved"])
numbers["S11.62"] = round(ours["odds_ratio"], 2)
numbers["S11.63"] = round(ours["ci_low"], 2)
numbers["S11.64"] = round(ours["ci_high"], 2)
numbers["S11.65"] = float(ours["p_value"])

theirs = enrichment.or_rs(pharmaprojects["geneticSupport_old"] == 1, pharmaprojects["approved"])
numbers["S11.69"] = round(theirs["odds_ratio"], 2)
numbers["S11.70"] = round(theirs["ci_low"], 2)
numbers["S11.71"] = round(theirs["ci_high"], 2)
# The published P is the Wald test on the log odds ratio, which is the P that goes with
# the Woolf interval reported beside it: 1.8e-20. Fisher's exact on the same table gives
# 1.4e-18. Both are printed.
numbers["S11.72"] = float(theirs["z_p_value"])
numbers["S11.73"] = round(theirs["relative_success"], 2)

ours_mask = enrichment.support_mask(pharmaprojects)
theirs_mask = pharmaprojects["geneticSupport_old"] == 1
numbers["S11.66"] = int((ours_mask & theirs_mask).sum())
numbers["S11.67"] = int(ours_mask.sum())
numbers["S11.68"] = int(theirs_mask.sum())
print(f"our support: OR {numbers['S11.62']} [{numbers['S11.63']}, {numbers['S11.64']}], P {numbers['S11.65']:.1e}")
print(
    f"their annotation: OR {numbers['S11.69']} [{numbers['S11.70']}, {numbers['S11.71']}], "
    f"P {numbers['S11.72']:.1e} (Wald on log OR; Fisher gives "
    f"{theirs['p_value']:.1e}), RS {numbers['S11.73']}"
)
print(f"the two overlap on {numbers['S11.66']} pairs, of {numbers['S11.67']} and {numbers['S11.68']}")

our support: OR 1.65 [1.3, 2.11], P 1.1e-04
their annotation: OR 2.32 [1.94, 2.78], P 1.8e-20 (Wald on log OR; Fisher gives 1.4e-18), RS 2.03
the two overlap on 287 pairs, of 469 and 858


## Stratified contrasts

Each stratum against the unsupported pairs only, with the difference between two supported strata
tested by the contrast of Methods "Clinical trials success modelling".

In [4]:
def stratum(frame, mask, label):
    """Enrichment of one supported stratum against the unsupported pairs."""
    none = frame["support_all"] == 0
    subset = frame[mask | none]
    return {"stratum": label, **enrichment.or_rs(mask.loc[subset.index], subset["approved"])}


def difference(low, high):
    """Contrast between two strata, on the log-odds scale."""
    return enrichment.contrast(
        low["yes_evid-high_clinphase"],
        low["yes_evid-low_clinphase"],
        high["yes_evid-high_clinphase"],
        high["yes_evid-low_clinphase"],
    )


rows = []
for label, frame in DATASETS:
    pav = frame["support_pav"] == 1
    non_pav = (frame["support_all"] == 1) & (frame["support_pav"] == 0)
    low_gps = (frame["support_all"] == 1) & frame["in_gps"] & (frame["gps"] <= 5)
    high_gps = (frame["support_all"] == 1) & frame["in_gps"] & (frame["gps"] >= 10)
    for name, mask in [("PAV", pav), ("non-PAV", non_pav), ("gPS <= 5", low_gps), ("gPS >= 10", high_gps)]:
        rows.append({"dataset": label, **stratum(frame, mask, name)})
strata = pd.DataFrame(rows).set_index(["dataset", "stratum"])

contrasts = []
for label, _ in DATASETS:
    for low_name, high_name in [("PAV", "non-PAV"), ("gPS <= 5", "gPS >= 10")]:
        low, high = strata.loc[(label, low_name)], strata.loc[(label, high_name)]
        test = difference(low, high)
        contrasts.append(
            {
                "dataset": label,
                "comparison": f"{low_name} vs {high_name}",
                "or_low": low["odds_ratio"],
                "or_high": high["odds_ratio"],
                "ratio": low["odds_ratio"] / high["odds_ratio"],
                "P": test["p_value"],
            }
        )
contrast_table = pd.DataFrame(contrasts)
contrast_table.round(4)

,dataset,comparison,or_low,or_high,ratio,P
0,ChEMBL,PAV vs non-PAV,6.0483,3.0924,1.9558,0.0002
1,ChEMBL,gPS <= 5 vs gPS >= 10,4.7983,2.9677,1.6168,0.0077
2,Pharmaprojects,PAV vs non-PAV,2.3382,1.3998,1.6704,0.0399
3,Pharmaprojects,gPS <= 5 vs gPS >= 10,1.9190,1.6225,1.1827,0.5423


In [5]:
def contrast_value(dataset, comparison, column):
    """One cell of the contrast table."""
    row = contrast_table[(contrast_table["dataset"] == dataset) & (contrast_table["comparison"] == comparison)].iloc[0]
    return float(row[column])


numbers["S11.74"] = round(contrast_value("Pharmaprojects", "PAV vs non-PAV", "or_low"), 2)
numbers["S11.77"] = round(contrast_value("Pharmaprojects", "PAV vs non-PAV", "or_high"), 2)
numbers["S11.80"] = round(contrast_value("Pharmaprojects", "PAV vs non-PAV", "P"), 3)
numbers["S11.81"] = round(contrast_value("Pharmaprojects", "gPS <= 5 vs gPS >= 10", "or_low"), 2)
numbers["S11.82"] = round(contrast_value("Pharmaprojects", "gPS <= 5 vs gPS >= 10", "or_high"), 2)
numbers["S11.83"] = round(contrast_value("Pharmaprojects", "gPS <= 5 vs gPS >= 10", "ratio"), 2)
numbers["S11.84"] = round(contrast_value("Pharmaprojects", "gPS <= 5 vs gPS >= 10", "P"), 2)
numbers["S11.85"] = round(contrast_value("ChEMBL", "gPS <= 5 vs gPS >= 10", "or_low"), 2)
numbers["S11.86"] = round(contrast_value("ChEMBL", "gPS <= 5 vs gPS >= 10", "or_high"), 2)
numbers["S11.87"] = round(contrast_value("ChEMBL", "gPS <= 5 vs gPS >= 10", "ratio"), 2)
numbers["S11.88"] = round(contrast_value("ChEMBL", "gPS <= 5 vs gPS >= 10", "P"), 4)
print({k: numbers[k] for k in ["S11.74", "S11.77", "S11.80", "S11.81", "S11.82", "S11.83", "S11.84"]})
print({k: numbers[k] for k in ["S11.85", "S11.86", "S11.87", "S11.88"]})

{'S11.74': 2.34, 'S11.77': 1.4, 'S11.80': 0.04, 'S11.81': 1.92, 'S11.82': 1.62, 'S11.83': 1.18, 'S11.84': 0.54}
{'S11.85': 4.8, 'S11.86': 2.97, 'S11.87': 1.62, 'S11.88': 0.0077}


## The criterion applied unchanged

In [6]:
applied = enrichment.or_rs(
    enrichment.support_mask(pharmaprojects, pav=True, ta_min=2, ta_max=5), pharmaprojects["approved"]
)
baseline_pp = enrichment.or_rs(enrichment.support_mask(pharmaprojects), pharmaprojects["approved"])
baseline_chembl = enrichment.or_rs(enrichment.support_mask(chembl), chembl["approved"])
criterion_chembl = enrichment.or_rs(enrichment.support_mask(chembl, pav=True, ta_min=2, ta_max=5), chembl["approved"])

numbers["S11.89"] = round(applied["odds_ratio"], 2)
numbers["S11.90"] = round(applied["ci_low"], 2)
numbers["S11.91"] = round(applied["ci_high"], 2)
numbers["S11.92"] = round(applied["relative_success"], 2)
numbers["S11.93"] = float(applied["p_value"])
numbers["S11.94"] = applied["yes_evid-high_clinphase"]
numbers["S11.95"] = applied["n_support"]
numbers["S11.96"] = round(applied["odds_ratio"] / baseline_pp["odds_ratio"], 2)
numbers["S11.97"] = round(criterion_chembl["odds_ratio"] / baseline_chembl["odds_ratio"], 2)
print(
    f"criterion in Pharmaprojects: OR {numbers['S11.89']} [{numbers['S11.90']}, {numbers['S11.91']}], "
    f"RS {numbers['S11.92']}, P {numbers['S11.93']:.1e}"
)
print(f"{numbers['S11.94']} launched of {numbers['S11.95']} supported pairs")
print(f"lift over each resource's own baseline: {numbers['S11.96']} here, {numbers['S11.97']} in ChEMBL")

criterion in Pharmaprojects: OR 4.5 [2.66, 7.6], RS 3.16, P 2.5e-07
23 launched of 60 supported pairs
lift over each resource's own baseline: 2.72 here, 2.84 in ChEMBL


## Non-linearity, and the power to detect it

The power calculation is stated before the test: how often would the quadratic likelihood-ratio
test reach P < 0.05 on the Pharmaprojects design if the effect were as strong as in ChEMBL, and if
it were scaled down by the attenuation the resource shows on genetic support alone.

In [7]:
def nonlinear(frame, label):
    """Nested logistic fits for the quadratic log(TA+1) term, on one resource."""
    design = pd.DataFrame(
        {
            "outcome": frame["approved"].to_numpy(),
            "geneticSupport": frame["support_all"].to_numpy(),
            "logta": np.log(frame["ta"].to_numpy() + 1),
        }
    )
    design["logta2"] = design["logta"] ** 2
    baseline = smf.logit("outcome ~ geneticSupport", data=design).fit(disp=False)
    linear = smf.logit("outcome ~ geneticSupport + logta", data=design).fit(disp=False)
    quadratic = smf.logit("outcome ~ geneticSupport + logta + logta2", data=design).fit(disp=False)
    statistic = 2 * (float(quadratic.llf) - float(linear.llf))
    interval = quadratic.conf_int().loc["logta2"].to_numpy()
    return (
        {
            "dataset": label,
            "LR": statistic,
            "P": float(chi2.sf(statistic, 1)),
            "coefficient": float(quadratic.params["logta2"]),
            "ci_low": float(interval[0]),
            "ci_high": float(interval[1]),
        },
        quadratic,
        design,
    )


chembl_result, chembl_model, _ = nonlinear(chembl, "ChEMBL")
pp_result, pp_model, pp_design = nonlinear(pharmaprojects, "Pharmaprojects")
pd.DataFrame([chembl_result, pp_result]).round(4)

,dataset,LR,P,coefficient,ci_low,ci_high
0,ChEMBL,64.8973,0.0000,-0.2667,-0.3331,-0.2002
1,Pharmaprojects,0.4661,0.4948,-0.0424,-0.1643,0.0795


In [8]:
attenuation = float(np.log(baseline_pp["odds_ratio"]) / np.log(baseline_chembl["odds_ratio"]))
slopes = chembl_model.params
observed_rate = float(pharmaprojects["approved"].mean())


def power(scale, simulations=500, seed=SEED):
    """Share of simulated Pharmaprojects datasets where the quadratic test reaches P < 0.05."""
    generator = np.random.default_rng(seed)
    linear_predictor = scale * (
        slopes["geneticSupport"] * pp_design["geneticSupport"]
        + slopes["logta"] * pp_design["logta"]
        + slopes["logta2"] * pp_design["logta2"]
    )
    # centre the intercept so the simulated approval rate matches the resource's own
    intercept = np.log(observed_rate / (1 - observed_rate)) - float(linear_predictor.mean())
    probability = 1 / (1 + np.exp(-(intercept + linear_predictor)))
    hits = 0
    for _ in range(simulations):
        simulated = pp_design.assign(outcome=generator.binomial(1, probability))
        linear = smf.logit("outcome ~ geneticSupport + logta", data=simulated).fit(disp=False)
        quadratic = smf.logit("outcome ~ geneticSupport + logta + logta2", data=simulated).fit(disp=False)
        if chi2.sf(2 * (float(quadratic.llf) - float(linear.llf)), 1) < 0.05:
            hits += 1
    return 100 * hits / simulations


numbers["S11.98"] = round(pp_result["LR"], 2)
numbers["S11.99"] = round(pp_result["P"], 2)
numbers["S11.102"] = round(attenuation, 3)
numbers["S11.103"] = round(pp_result["coefficient"], 3)
numbers["S11.104"] = round(pp_result["ci_low"], 3)
numbers["S11.105"] = round(pp_result["ci_high"], 3)
numbers["S11.101"] = round(power(1.0), 0)
numbers["S11.100"] = round(power(attenuation), 0)
print(
    f"quadratic term in Pharmaprojects: LR {numbers['S11.98']}, P {numbers['S11.99']}, "
    f"coefficient {numbers['S11.103']} [{numbers['S11.104']}, {numbers['S11.105']}]"
)
print(
    f"attenuation {numbers['S11.102']} | power at full strength {numbers['S11.101']}%, "
    f"attenuation-matched {numbers['S11.100']}%"
)

quadratic term in Pharmaprojects: LR 0.47, P 0.49, coefficient -0.042 [-0.164, 0.08]
attenuation 0.392 | power at full strength 99.0%, attenuation-matched 36.0%


## The pleiotropy ceiling

Supported pairs at 2-5 therapeutic areas against those at six or more, each against the unsupported
pairs; single-therapeutic-area pairs are left out.

In [9]:
def outside_both(frame, mask, other, label):
    """The same enrichment with every pair outside both strata as the reference.

    This is the reference the published PAV rows of the ceiling table used, as opposed to the
    unsupported pairs only. It leaves the ratio of the two odds ratios unchanged, because both move
    by the same factor, and it is reported here only so the published values are traceable.
    """
    subset = frame[mask | ~(mask | other)]
    return {"stratum": label, **enrichment.or_rs(mask.loc[subset.index], subset["approved"])}


def ceiling(frame, dataset, support_column, support_label):
    """Low versus high pleiotropy among supported pairs, against the unsupported reference."""
    supported = frame[support_column] == 1
    low = supported & frame["in_gps"] & (frame["ta"] >= 2) & (frame["ta"] < 6)
    high = supported & frame["in_gps"] & (frame["ta"] >= 6)
    low_result = stratum(frame, low, "low")
    high_result = stratum(frame, high, "high")
    test = difference(low_result, high_result)
    return {
        "dataset": dataset,
        "support": support_label,
        "or_low": low_result["odds_ratio"],
        "or_high": high_result["odds_ratio"],
        "n_low": low_result["n_support"],
        "approved_low": low_result["yes_evid-high_clinphase"],
        "n_high": high_result["n_support"],
        "approved_high": high_result["yes_evid-high_clinphase"],
        "ratio": low_result["odds_ratio"] / high_result["odds_ratio"],
        "P": test["p_value"],
        "or_low_outside_both": outside_both(frame, low, high, "low")["odds_ratio"],
        "or_high_outside_both": outside_both(frame, high, low, "high")["odds_ratio"],
    }


ceiling_table = pd.DataFrame(
    [
        ceiling(chembl, "ChEMBL", "support_pav", "PAV"),
        ceiling(pharmaprojects, "Pharmaprojects", "support_pav", "PAV"),
        ceiling(chembl, "ChEMBL", "support_all", "Any"),
        ceiling(pharmaprojects, "Pharmaprojects", "support_all", "Any"),
    ]
)
for offset, (_, row) in enumerate(ceiling_table.iterrows()):
    base = 106 + 8 * offset
    numbers[f"S11.{base}"] = round(float(row["or_low"]), 2)
    numbers[f"S11.{base + 1}"] = round(float(row["or_high"]), 2)
    numbers[f"S11.{base + 2}"] = round(float(row["ratio"]), 2)
    numbers[f"S11.{base + 3}"] = float(row["P"])
    numbers[f"S11.{base + 4}"] = int(row["n_low"])
    numbers[f"S11.{base + 5}"] = int(row["approved_low"])
    numbers[f"S11.{base + 6}"] = int(row["n_high"])
    numbers[f"S11.{base + 7}"] = int(row["approved_high"])
ceiling_table.round(4)

,dataset,support,or_low,or_high,n_low,approved_low,n_high,approved_high,ratio,P,or_low_outside_both,or_high_outside_both
0,ChEMBL,PAV,10.5916,3.1814,87,51,67,20,3.3292,0.0005,10.3199,3.0999
1,Pharmaprojects,PAV,4.5806,1.1053,60,23,69,9,4.1441,0.0014,4.5016,1.0863
2,ChEMBL,Any,4.0124,2.9686,398,139,285,81,1.3516,0.0733,3.9967,2.9569
3,Pharmaprojects,Any,1.8765,1.4814,202,41,233,39,1.2668,0.3399,1.8716,1.4774


### Which reference the published table used

The published ceiling table is not internally consistent about its reference group, and the two
right-hand columns above show it. Its **PAV** rows match each stratum against every pair *outside
both strata* — 10.32 and 3.10 in ChEMBL, 4.50 and 1.09 in Pharmaprojects, to the digit. Its
**any-support** rows match the unsupported pairs only — 4.01 and 2.97, 1.88 and 1.48.

Everything else in this section, and the stratified analysis in the main text, uses the unsupported
pairs as the reference, so that is what is registered here for all four rows. The choice moves both
odds ratios in a row by the same factor, so the ratio and the difference test are identical either
way, which is why those columns reproduce regardless.

## The interaction between PAV support and pleiotropy

On supported pairs only: `outcome ~ PAV + low + PAV × low`, where low marks 2-5 therapeutic areas
and single-area pairs are left out. The permutation test shuffles the pleiotropy label within the
PAV and non-PAV groups, which holds both main effects and all four group sizes fixed.

In [10]:
def interaction(frame, label, permutations=PERMUTATIONS, seed=SEED):
    """PAV by pleiotropy interaction on approval, with a likelihood-ratio and a permutation P."""
    supported = (frame["support_all"] == 1) & frame["in_gps"] & (frame["ta"] >= 2)
    data = pd.DataFrame(
        {
            "outcome": frame.loc[supported, "approved"].to_numpy(),
            "pav": frame.loc[supported, "support_pav"].to_numpy(),
            "low": (frame.loc[supported, "ta"] < 6).astype(int).to_numpy(),
        }
    )
    full = smf.logit("outcome ~ pav + low + pav:low", data=data).fit(disp=False)
    reduced = smf.logit("outcome ~ pav + low", data=data).fit(disp=False)
    statistic = 2 * (float(full.llf) - float(reduced.llf))
    interval = np.exp(full.conf_int().loc["pav:low"].to_numpy())

    observed = float(full.params["pav:low"])
    generator = np.random.default_rng(seed)
    extreme = 0
    for _ in range(permutations):
        shuffled = data.copy()
        for value in (0, 1):
            rows = shuffled["pav"] == value
            shuffled.loc[rows, "low"] = generator.permutation(shuffled.loc[rows, "low"].to_numpy())
        try:
            fitted = smf.logit("outcome ~ pav + low + pav:low", data=shuffled).fit(disp=False)
        except Exception:
            continue
        if abs(float(fitted.params["pav:low"])) >= abs(observed):
            extreme += 1
    rates = data.groupby(["low", "pav"])["outcome"].mean()
    return {
        "dataset": label,
        "interaction_or": float(np.exp(observed)),
        "ci_low": float(interval[0]),
        "ci_high": float(interval[1]),
        "lrt_p": float(chi2.sf(statistic, 1)),
        "permutation_p": (extreme + 1) / (permutations + 1),
        "pairs": len(data),
        "approved": int(data["outcome"].sum()),
        "rate_low_no_pav": 100 * float(rates.loc[(1, 0)]),
        "rate_low_pav": 100 * float(rates.loc[(1, 1)]),
        "rate_high_no_pav": 100 * float(rates.loc[(0, 0)]),
        "rate_high_pav": 100 * float(rates.loc[(0, 1)]),
    }


interactions = pd.DataFrame([interaction(chembl, "ChEMBL"), interaction(pharmaprojects, "Pharmaprojects")])
interactions.round(4)

,dataset,interaction_or,ci_low,ci_high,lrt_p,permutation_p,pairs,approved,rate_low_no_pav,rate_low_pav,rate_high_no_pav,rate_high_pav
0,ChEMBL,3.2778,1.5067,7.1311,0.0024,0.0018,683,220,28.2958,58.6207,27.9817,29.8507
1,Pharmaprojects,6.3915,2.1741,18.7895,0.0005,0.0010,435,80,12.6761,38.3333,18.2927,13.0435


In [11]:
for offset, (_, row) in enumerate(interactions.iterrows()):
    base = 138 + 11 * offset
    numbers[f"S11.{base}"] = round(float(row["interaction_or"]), 2)
    numbers[f"S11.{base + 1}"] = round(float(row["ci_low"]), 2)
    numbers[f"S11.{base + 2}"] = round(float(row["ci_high"]), 2)
    numbers[f"S11.{base + 3}"] = float(row["lrt_p"])
    numbers[f"S11.{base + 4}"] = float(row["permutation_p"])
    numbers[f"S11.{base + 5}"] = int(row["pairs"])
    numbers[f"S11.{base + 6}"] = int(row["approved"])
    numbers[f"S11.{base + 7}"] = round(float(row["rate_low_no_pav"]), 1)
    numbers[f"S11.{base + 8}"] = round(float(row["rate_low_pav"]), 1)
    numbers[f"S11.{base + 9}"] = round(float(row["rate_high_no_pav"]), 1)
    numbers[f"S11.{base + 10}"] = round(float(row["rate_high_pav"]), 1)
print(
    interactions[["dataset", "interaction_or", "lrt_p", "permutation_p", "pairs", "approved"]]
    .round(4)
    .to_string(index=False)
)

       dataset  interaction_or  lrt_p  permutation_p  pairs  approved
        ChEMBL          3.2778 0.0024         0.0018    683       220
Pharmaprojects          6.3915 0.0005         0.0010    435        80


## Write the results

In [12]:
print(paper.save_results("sr11_external", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr11_external.json


,computed
S11.55,7390.0
S11.56,2274.0
S11.57,3303.0
S11.58,900.0
S11.59,913.0
...,...
S11.155,80.0
S11.156,12.7
S11.157,38.3
S11.158,18.3
